In [1]:
!pip install opencv-python pillow imagehash pandas tqdm

  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached ImageHash-4.3.2-py2.py3-none-any.whl.metadata (8.4 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
Using cached ImageHash-4.3.2-py2.py3-none-any.whl (296 kB)

   ---------------------------------------- 0/2 [opencv-python]
   ---------------------------------------- 0/2 [opencv-python]
   ---------------------------------------- 0/2 [opencv-python]
   ---------------------------------------- 2/2 [imagehash]



In [3]:
import os
from PIL import Image
import imagehash
import pandas as pd
from tqdm import tqdm

# Tự động lấy thư mục gốc và trỏ đến data/raw
BASE_DIR = os.getcwd()
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
OUTPUT_CSV = os.path.join(BASE_DIR, "data", "metadata.csv")

metadata_list = []
hashes = {}

print(f"Đang quét thư mục dữ liệu gốc tại: {RAW_DIR}")

if not os.path.exists(RAW_DIR):
    print(f"[LỖI] Thư mục {RAW_DIR} không tồn tại! Bạn hãy kiểm tra lại đường dẫn.")
else:
    all_files = []
    for root, dirs, files in os.walk(RAW_DIR):
        for file in files:
            all_files.append(os.path.join(root, file))
            
    print(f"-> Tổng số file tìm thấy trong thư mục raw (tính cả file khác): {len(all_files)}")

    for file_path in tqdm(all_files):
        if file_path.lower().endswith(('.jpg', '.jpeg', '.png')):
            file = os.path.basename(file_path)
            
            # 1. Kiểm tra ảnh hỏng
            try:
                pil_img = Image.open(file_path)
                pil_img.verify()
                pil_img = Image.open(file_path) # Mở lại để lấy size sau khi verify
                width, height = pil_img.size
            except Exception:
                print(f"\n[CẢNH BÁO] Ảnh hỏng hoặc lỗi định dạng: {file_path}")
                continue
            
            # 2. Kiểm tra ảnh trùng lặp bằng pHash
            try:
                img_hash = str(imagehash.phash(pil_img))
                if img_hash in hashes:
                    continue
                hashes[img_hash] = file_path
            except Exception:
                img_hash = "unknown"

            # 3. Trích xuất nhãn bệnh từ tên thư mục cha
            # Cấu trúc kỳ vọng: data/raw/[Tên_Bệnh]/[Tên_File].jpg
            parts = os.path.normpath(file_path).split(os.sep)
            disease_label = parts[-2] if len(parts) >= 2 else "Unknown"
            
            filename_no_ext = os.path.splitext(file)[0]
            specimen_id = filename_no_ext.split('_')[0] if '_' in filename_no_ext else "unknown_specimen"

            metadata_list.append({
                "image_id": file,
                "specimen_id": specimen_id,
                "disease_label": disease_label,
                "file_path": file_path,
                "width": width,
                "height": height,
                "phash": img_hash
            })

# Kiểm tra trước khi tạo DataFrame
if len(metadata_list) > 0:
    df_meta = pd.DataFrame(metadata_list)
    os.makedirs(os.path.join(BASE_DIR, "data"), exist_ok=True)
    df_meta.to_csv(OUTPUT_CSV, index=False)
    print(f"\nĐã hoàn tất! Tổng số ảnh hợp lệ sau khi làm sạch: {len(df_meta)}")
    print("Phân bố số lượng theo nhãn bệnh:")
    print(df_meta['disease_label'].value_counts())
else:
    print("\n[LỖI NGHIÊM TRỌNG] Không tìm thấy bất kỳ file ảnh hợp lệ nào! Hãy chắc chắn bạn đã giải nén dataset MatsyaDx-BD vào thư mục 'data/raw/' theo đúng cấu trúc phân loại theo thư mục bệnh.")

Đang quét thư mục dữ liệu gốc tại: D:\Project\scripts\data\raw
[LỖI] Thư mục D:\Project\scripts\data\raw không tồn tại! Bạn hãy kiểm tra lại đường dẫn.

[LỖI NGHIÊM TRỌNG] Không tìm thấy bất kỳ file ảnh hợp lệ nào! Hãy chắc chắn bạn đã giải nén dataset MatsyaDx-BD vào thư mục 'data/raw/' theo đúng cấu trúc phân loại theo thư mục bệnh.


In [2]:
import os
from PIL import Image
import imagehash
import pandas as pd
from tqdm import tqdm

RAW_DIR = r"D:\Project\data\raw"  
# Dùng tên file mới hoàn toàn để không bao giờ bị trùng khóa với file cũ bị lỗi
OUTPUT_CSV = r"D:\Project\data\metadata_v2.csv"

metadata_list = []
hashes = {}

print(f"Đang quét thư mục dữ liệu gốc tại: {RAW_DIR}")

if not os.path.exists(RAW_DIR):
    print(f"[LỖI] Thư mục {RAW_DIR} không tồn tại!")
else:
    all_files = []
    for root, dirs, files in os.walk(RAW_DIR):
        for file in files:
            all_files.append(os.path.join(root, file))
            
    print(f"-> Tổng số file tìm thấy: {len(all_files)}")

    for file_path in tqdm(all_files):
        if file_path.lower().endswith(('.jpg', '.jpeg', '.png')):
            file = os.path.basename(file_path)
            
            try:
                pil_img = Image.open(file_path)
                pil_img.verify()
                pil_img = Image.open(file_path)
                width, height = pil_img.size
            except Exception:
                continue
            
            try:
                img_hash = str(imagehash.phash(pil_img))
                if img_hash in hashes:
                    continue
                hashes[img_hash] = file_path
            except Exception:
                img_hash = "unknown"

            parts = os.path.normpath(file_path).split(os.sep)
            disease_label = parts[-2] if len(parts) >= 2 else "Unknown"
            
            filename_no_ext = os.path.splitext(file)[0]
            specimen_id = filename_no_ext.split('_')[0] if '_' in filename_no_ext else "unknown_specimen"

            metadata_list.append({
                "image_id": file,
                "specimen_id": specimen_id,
                "disease_label": disease_label,
                "file_path": file_path,
                "width": width,
                "height": height,
                "phash": img_hash
            })

if len(metadata_list) > 0:
    df_meta = pd.DataFrame(metadata_list)
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    
    # Dùng Python thuần để ghi file an toàn, tránh hoàn toàn lỗi PermissionError của pandas
    csv_content = df_meta.to_csv(index=False)
    with open(OUTPUT_CSV, "w", encoding="utf-8") as f:
        f.write(csv_content)
        
    print(f"\nĐã hoàn tất! Tổng số ảnh hợp lệ: {len(df_meta)}")
    print(f"Đã lưu thành công tại: {OUTPUT_CSV}")
    print("Phân bố số lượng theo nhãn bệnh:")
    print(df_meta['disease_label'].value_counts())
else:
    print("\n[LỖI] Không tìm thấy file ảnh hợp lệ nào!")

Đang quét thư mục dữ liệu gốc tại: D:\Project\data\raw
-> Tổng số file tìm thấy: 2139


100%|██████████| 2139/2139 [05:22<00:00,  6.63it/s]



Đã hoàn tất! Tổng số ảnh hợp lệ: 1939
Đã lưu thành công tại: D:\Project\data\metadata_v2.csv
Phân bố số lượng theo nhãn bệnh:
disease_label
Fish_82    97
Fish_83    93
Fish_85    90
Fish_80    88
Fish_84    75
           ..
Fish_25     7
Fish_20     6
Fish_63     5
Fish_9      4
Fish_26     4
Name: count, Length: 86, dtype: int64


In [1]:
import pandas as pd
import os

METADATA_PATH = r"D:\Project\data\metadata_v2.csv"
SPLITS_DIR = r"D:\Project\data\splits"

if not os.path.exists(METADATA_PATH):
    print(f"[LỖI] Không tìm thấy file {METADATA_PATH}!")
else:
    # 1. Đọc file metadata
    df = pd.read_csv(METADATA_PATH)
    
    train_list = []
    val_list = []
    test_list = []
    
    # 2. Chia đều từng cá thể theo từng nhãn bệnh để đảm bảo cân bằng và không bị lỗi
    for disease, group in df.groupby('disease_label'):
        # Lấy danh sách các cá thể (specimen_id) duy nhất trong bệnh này
        unique_specimens = group['specimen_id'].unique()
        n_specimens = len(unique_specimens)
        
        # Tính số lượng cá thể cho train, val, test (70% - 15% - 15%)
        n_test = max(1, int(n_specimens * 0.15))
        n_val = max(1, int(n_specimens * 0.15))
        n_train = n_specimens - n_test - n_val
        
        # Nếu số lượng quá ít, đảm bảo ít nhất có 1 con cho train
        if n_train < 1:
            n_train = 1
            
        test_specimens = unique_specimens[:n_test]
        val_specimens = unique_specimens[n_test:n_test + n_val]
        train_specimens = unique_specimens[n_test + n_val:]
        if len(train_specimens) == 0:  # Phòng hờ trường hợp quá ít cá thể
            train_specimens = val_specimens
            val_specimens = test_specimens
            
        # Phân chia dòng dữ liệu ảnh dựa vào tập cá thể tương ứng
        train_list.append(group[group['specimen_id'].isin(train_specimens)])
        val_list.append(group[group['specimen_id'].isin(val_specimens)])
        test_list.append(group[group['specimen_id'].isin(test_specimens)])

    # Gộp dữ liệu lại
    df_train = pd.concat(train_list, ignore_index=True)
    df_val = pd.concat(val_list, ignore_index=True)
    df_test = pd.concat(test_list, ignore_index=True)
    
    df_train['split'] = 'train'
    df_val['split'] = 'val'
    df_test['split'] = 'test'
    
    # Gộp chung vào DataFrame tổng để lưu file
    df_final = pd.concat([df_train, df_val, df_test], ignore_index=True)

    # 3. Lưu kết quả ra 3 file CSV trong thư mục splits
    os.makedirs(SPLITS_DIR, exist_ok=True)
    
    df_train.to_csv(os.path.join(SPLITS_DIR, "train.csv"), index=False)
    df_val.to_csv(os.path.join(SPLITS_DIR, "val.csv"), index=False)
    df_test.to_csv(os.path.join(SPLITS_DIR, "test.csv"), index=False)

    print("Phân chia dữ liệu cá thể thành công xuất sắc và an toàn tuyệt đối!")
    print(f"- Số lượng ảnh Train: {len(df_train)}")
    print(f"- Số lượng ảnh Val:   {len(df_val)}")
    print(f"- Số lượng ảnh Test:  {len(df_test)}")
    print(f"Các file phân chia đã được lưu tại thư mục: {SPLITS_DIR}")

Phân chia dữ liệu cá thể thành công xuất sắc và an toàn tuyệt đối!
- Số lượng ảnh Train: 49
- Số lượng ảnh Val:   1024
- Số lượng ảnh Test:  1872
Các file phân chia đã được lưu tại thư mục: D:\Project\data\splits
